# SentinelAI — 03. Behavioral Anomaly Detection & Benchmarking

**Unit**: Application Security and Intrusion Detection  
**Phase**: Phase 9 — Behavioral Anomaly Detection  
**Target Architecture**: Unsupervised Isolation Forest vs One-Class SVM  
**Feature Space**: 6-dimensional behavioral telemetry vectors (velocity, burst frequency, failed auth, 4xx error rate, path entropy, request interval)


In [1]:
import os
import time
import json
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

# Load dataset
df = pd.read_csv('../datasets/processed/behavioral_telemetry_dataset.csv')
print(f'Total records: {len(df):,}')
print(df['attack_type'].value_counts())
df.head()

## Feature Scaling & Train/Test Partitioning

Unsupervised models are trained on telemetry profiles to isolate anomalous outliers.

In [2]:
feature_cols = [
    'request_frequency',
    'burst_frequency',
    'failed_auth_count',
    'error_4xx_rate',
    'path_entropy',
    'avg_interval_ms',
]

train_df = df.iloc[:9000].copy()
test_df = df.iloc[9000:].copy()

X_train = train_df[feature_cols].values
X_test = test_df[feature_cols].values
y_test = test_df['is_anomaly'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train samples: {len(X_train_scaled):,}, Test samples: {len(X_test_scaled):,}')
print(f'Anomalies in test set: {y_test.sum():,} ({y_test.sum()/len(y_test):.1%})')

## Model 1: Isolation Forest Training & Benchmark

Isolation Forest recursively partitions feature space using random split planes. Anomalies require significantly fewer splits to isolate than clustered normal traffic.

In [3]:
iso_forest = IsolationForest(
    n_estimators=150,
    contamination=0.10,
    random_state=42,
    n_jobs=-1,
)

t0 = time.time()
iso_forest.fit(X_train_scaled)
train_time = time.time() - t0

t0 = time.time()
raw_preds = iso_forest.predict(X_test_scaled)
infer_latency_us = (time.time() - t0) / len(X_test_scaled) * 1e6

binary_preds = np.where(raw_preds == -1, 1, 0)
scores = -iso_forest.decision_function(X_test_scaled)

print('=== Isolation Forest Benchmark ===')
print(f'Train time:        {train_time:.3f} s')
print(f'Inference latency: {infer_latency_us:.2f} µs / sample')
print(f'Precision:         {precision_score(y_test, binary_preds):.4f}')
print(f'Recall:            {recall_score(y_test, binary_preds):.4f}')
print(f'F1-Score:          {f1_score(y_test, binary_preds):.4f}')
print(f'ROC-AUC:           {roc_auc_score(y_test, scores):.4f}')

## Model 2: One-Class SVM Comparison

One-Class SVM attempts to wrap a high-dimensional hyperplane around normal points.

In [4]:
oc_svm = OneClassSVM(kernel='rbf', nu=0.10, gamma='scale')

t0 = time.time()
oc_svm.fit(X_train_scaled[:5000])
svm_train_time = time.time() - t0

t0 = time.time()
svm_preds = oc_svm.predict(X_test_scaled)
svm_latency_us = (time.time() - t0) / len(X_test_scaled) * 1e6

svm_binary = np.where(svm_preds == -1, 1, 0)
svm_scores = -oc_svm.decision_function(X_test_scaled)

print('=== One-Class SVM Benchmark ===')
print(f'Train time:        {svm_train_time:.3f} s')
print(f'Inference latency: {svm_latency_us:.2f} µs / sample')
print(f'Precision:         {precision_score(y_test, svm_binary):.4f}')
print(f'Recall:            {recall_score(y_test, svm_binary):.4f}')
print(f'F1-Score:          {f1_score(y_test, svm_binary):.4f}')
print(f'ROC-AUC:           {roc_auc_score(y_test, svm_scores):.4f}')

## Architectural Evaluation & Decision

| Metric | Isolation Forest | One-Class SVM | Impact |
|---|---|---|---|
| **ROC-AUC** | **99.92%** | 80.04% | Isolation Forest separates distributions with near-perfection |
| **Precision** | **100.00%** | 81.52% | Zero false alarms on test set |
| **Inference Latency** | **~6.5 µs** | ~25.4 µs | 4x faster inline request execution |
| **Computational Scaling** | **O(t · n · log n)** | O(n³) | Isolation Forest scales linearly with traffic |

**Verdict**: Isolation Forest selected for SentinelAI production deployment.

In [5]:
# Verify serialized artifacts
loaded_model = joblib.load('../../ai-service/app/models/anomaly_detector/model.joblib')
loaded_scaler = joblib.load('../../ai-service/app/models/anomaly_detector/scaler.joblib')
loaded_meta = joblib.load('../../ai-service/app/models/anomaly_detector/meta.joblib')

print('Artifacts loaded successfully:')
print(f'Model type: {loaded_meta["model_type"]}')
print(f'Version:    {loaded_meta["version"]}')
print(f'Features:   {loaded_meta["features"]}')